Un uso extendido del filtrado homom ́orfico es la correcci ́on de iluminaci ́on no
uniforme en distintas zonas de la imagen, generalmente con alto contenido de
informaci ́on en la zona de bajo brillo. Por ejemplo, en filmaciones de c ́amaras de
seguridad y fotos con luz de d ́ıa con sol de frente. En im ́agenes de este tipo, el
filtro homom ́orfico corrige el contraste en la zona de inter ́es y acent ́ua los detalles
simult ́aneamente.
1. Genere la funci ́on de transferencia H que caracteriza a un filtro homom ́orfico.
2. Aplique el proceso en las im ́agenes ‘casilla.tif’ y ‘reunion.tif’, con
valores apropiados de gL, gH , D0 y orden (prueba y error en cada imagen...).
3. Verifique las bondades del m ́etodo comparando el resultado anterior con la
imagen que se obtiene al ecualizar la imagen original.
Esta t ́ecnica suele ser eficaz combinada con alguna t ́ecnica de manipulaci ́on
de histogramas, por ejemplo ecualizaci ́on. Ecualice el resultado del filtrado y
visual ́ıcelo junto a los dem ́as.

In [16]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from ipywidgets import interact, FloatSlider, IntSlider


Los defino en función de los filtros pasaaltos en la forma $\gama_H - \gamma_L  * PA(u,v) + \gamma_l$



In [17]:
def homomorphic_filter(img, gamma_l, gamma_h, c, D0):
    img_float = img.astype(np.float32)
    log_img = np.log(img_float + 1e-5)

    F = np.fft.fft2(log_img)
    F_shift = np.fft.fftshift(F)
    rows, cols = img.shape
    
    u = np.arange(rows) - rows // 2
    v = np.arange(cols) - cols // 2
    U, V = np.meshgrid(v, u)


    D = np.sqrt(U**2 + V**2)

    D0 = max(D0, 1e-5) 
    H = (gamma_h - gamma_l) * (1 - np.exp(-c * (D**2) / (D0**2))) + gamma_l
    G_shift = H * F_shift
    G = np.fft.ifftshift(G_shift)
    g = np.real(np.fft.ifft2(G))
    result = np.exp(g)

    result = cv2.normalize(
        result,
        None,
        alpha=0,
        beta=255,
        norm_type=cv2.NORM_MINMAX
    )

    return result.astype(np.uint8), H

def visualizar_homomorfico(img, gammaL=0.5, gammaH=2.0, c=1.0, D0=30):
    # Llamada corregida pasando los parámetros explícitos
    filtrada, H = homomorphic_filter(
        img,
        gamma_l=gammaL,
        gamma_h=gammaH,
        c=c,
        D0=D0
    )

    plt.figure(figsize=(18, 6))
    
    # ORIGINAL
    plt.subplot(1, 3, 1)
    plt.title("Imagen Original")
    plt.imshow(img, cmap='gray')
    plt.axis('off')

    # FILTRADA
    plt.subplot(1, 3, 2)
    plt.title(f"Filtro Homomórfico\n(gL={gammaL}, gH={gammaH}, D0={D0})")
    plt.imshow(filtrada, cmap='gray')
    plt.axis('off')

    # MÁSCARA H(u,v)
    plt.subplot(1, 3, 3)
    plt.title("Filtro H(u,v) en Frecuencia")
    plt.imshow(H, cmap='jet')
    plt.colorbar(fraction=0.046, pad=0.04) # Ajuste estético de la barra
    plt.axis('off')

    plt.tight_layout()
    plt.show()


In [18]:
# analisis de la mascara 
import numpy as np
import plotly.graph_objects as go

img = cv2.imread("../Imagenes_cursado/casilla.tif", cv2.IMREAD_GRAYSCALE)


rows, cols = img.shape

gamma_l = 0.5
gamma_h = 2.0
c = 1
D0 = 60

u = np.arange(rows)
v = np.arange(cols)

U, V = np.meshgrid(u, v, indexing='ij')

# distancia al centro
D = np.sqrt(
    (U - rows/2)**2 +
    (V - cols/2)**2
)

# ==========================================
# MÁSCARA HOMOMÓRFICA
# ==========================================

H = (
    (gamma_h - gamma_l)
    *
    (1 - np.exp(-c * (D**2)/(D0**2)))
) + gamma_l

# ==========================================
# PLOT 3D
# ==========================================

fig = go.Figure()

fig.add_trace(
    go.Surface(
        z=H,
        x=V,
        y=U
    )
)

# ==========================================
# CONFIGURACIÓN VISUAL
# ==========================================

fig.update_layout(
    title="Máscara Homomórfica H(u,v)",
    
    scene=dict(
        xaxis_title='Frecuencia horizontal (v)',
        yaxis_title='Frecuencia vertical (u)',
        zaxis_title='Ganancia H(u,v)',

        # cotas visuales útiles
        zaxis=dict(
            range=[gamma_l, gamma_h]
        )
    ),

    width=1000,
    height=800
)

fig.show()

# Analisis castilla

In [19]:
img = cv2.imread("../Imagenes_cursado/reunion.tif", cv2.IMREAD_GRAYSCALE)
interact(

    lambda gammaL, gammaH, c, D0:
        visualizar_homomorfico(
            img,
            gammaL,
            gammaH,
            c,
            D0
        ),

    gammaL=FloatSlider(
        value=0.5,
        min=0.0,
        max=2.0,
        step=0.1,
        description='γL'
    ),

    gammaH=FloatSlider(
        value=2.0,
        min=1.0,
        max=5.0,
        step=0.1,
        description='γH'
    ),

    c=FloatSlider(
        value=1.0,
        min=0.1,
        max=10.0,
        step=0.1,
        description='c'
    ),

    D0=IntSlider(
        value=30,
        min=1,
        max=200,
        step=1,
        description='D0'
    )
)


interactive(children=(FloatSlider(value=0.5, description='γL', max=2.0), FloatSlider(value=2.0, description='γ…

<function __main__.<lambda>(gammaL, gammaH, c, D0)>

Analisis de los parametros de nuestro filtro
- c controla la transicion. C chico la transicion es lenta y suave. C grande es abrupta y se parece mas al filtro ideal
- D_0 es nuestro radio, funciona como desvio
- y_l es la ganancia minima del filtro que las reciben las bajas frecuencias
- y_h es la gananciamaxima del filtro y las reciben las altas frecuencias

gamal =0
gamah = 1
c=10
d0 =3